In [ ]:
import os
import sys
import pyspark
import ctypes
import shutil
from delta import configure_spark_with_delta_pip
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql.window import Window
from datetime import datetime

# 1. Clear out any stale SparkContext references in Python
try:
    spark.stop()
except Exception:
    pass
pyspark.SparkContext._active_spark_context = None

# 2. Add Conda environment DLL directories so spawned Python workers don't crash on startup
conda_env_dir = os.path.dirname(sys.executable)
conda_lib_bin = os.path.join(conda_env_dir, "Library", "bin")
conda_scripts = os.path.join(conda_env_dir, "Scripts")

os.environ["PATH"] = f"C:\\hadoop\\bin;{conda_env_dir};{conda_lib_bin};{conda_scripts};" + os.environ["PATH"]
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["HADOOP_HOME"] = "C:\\hadoop"

# Force Windows to load hadoop.dll into process memory before Java boots
try:
    ctypes.WinDLL("C:\\hadoop\\bin\\hadoop.dll")
    print("SUCCESS: hadoop.dll loaded cleanly into Windows!")
except OSError as e:
    print(f"ERROR loading hadoop.dll: {e}")

java_opts = (
    "-Djava.library.path=C:\\hadoop\\bin "
    "--add-opens=java.base/java.lang=ALL-UNNAMED "
    "--add-opens=java.base/java.lang.invoke=ALL-UNNAMED "
    "--add-opens=java.base/java.lang.reflect=ALL-UNNAMED "
    "--add-opens=java.base/java.io=ALL-UNNAMED "
    "--add-opens=java.base/java.net=ALL-UNNAMED "
    "--add-opens=java.base/java.nio=ALL-UNNAMED "
    "--add-opens=java.base/java.util=ALL-UNNAMED "
    "--add-opens=java.base/java.util.concurrent=ALL-UNNAMED "
    "--add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED "
    "--add-opens=java.base/sun.nio.ch=ALL-UNNAMED "
    "--add-opens=java.base/sun.nio.cs=ALL-UNNAMED "
    "--add-opens=java.base/sun.security.action=ALL-UNNAMED "
    "--add-opens=java.base/sun.util.calendar=ALL-UNNAMED "
    "--add-opens=java.security.jgss/sun.security.krb5=ALL-UNNAMED "
    "-Djdk.security.auth.subject.useTL=true"
)

# 3. Force single-worker execution (local[1]) to prevent Windows socket exhaustion
builder = (
    pyspark.sql.SparkSession.builder.appName("LocalDeltaLakehouse")
    .master("local")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.driver.extraJavaOptions", java_opts)
    .config("spark.executor.extraJavaOptions", java_opts)
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.python.worker.reuse", "true")
    .config("spark.sql.shuffle.partitions", "1")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
try:
    # Explicitly force the JVM to load hadoop.dll from disk
    spark.sparkContext._jvm.java.lang.System.load("C:\\hadoop\\bin\\hadoop.dll")
    print("JVM successfully loaded hadoop.dll!")
except TypeError as e:
    print(f"Error: {e}")
print("Fresh SparkSession & Delta Lake ready!")

In [ ]:
# It is a one-time cleanup activity
"""
warehouse_db_dir = "C:/Users/Hethi/Downloads/spark-NA/spark-warehouse/dv2_silver.db"

# 1. Create the schema in your fresh Spark session first
spark.sql("CREATE SCHEMA IF NOT EXISTS dv2_silver")

# 2. Remove incomplete table folders left from previous interrupted runs
for table_folder in ["hub_loan_account", "link_customer_loan", "sat_loan_risk_status", "hub_collateral", "link_loan_collateral", "sat_collateral_valuation"]:
    folder_path = os.path.join(warehouse_db_dir, table_folder)
    if os.path.exists(folder_path):
        shutil.rmtree(folder_path)
        print(f"Cleaned up partial folder: {folder_path}")
    spark.sql(f"DROP TABLE IF EXISTS dv2_silver.{table_folder}")

print("Warehouse directory is clean. You can now re-run your Data Vault load cell!")
"""

In [ ]:
# uncomment the below only when you want to clean the tables from the dv2 path
"""
dv2_dir = "C:/users/Hethi/Downloads/spark-NA/dv2"
# 2. Remove incomplete table folders left from previous interrupted runs
for table_folder in ["hub_customer", "sat_customer_details","hub_loan_account", "link_customer_loan", "sat_loan_risk_status", "hub_collateral", "link_loan_collateral", "sat_collateral_valuation"]:
    folder_path = os.path.join(dv2_dir, table_folder)
    if os.path.exists(folder_path):
        shutil.rmtree(folder_path)
        print(f"Cleaned up partial folder: {folder_path}")
    spark.sql(f"DROP TABLE IF EXISTS dv2_silver.{table_folder}")

print("Warehouse directory is clean. You can now re-run your Data Vault load cell!")
"""

In [ ]:
# Define base dv2 path for Hub, Satellite, and Link Delta tables
dv2_path = "C:/users/Hethi/Downloads/spark-NA/dv2"

def prepare_staging_data(raw_data, load_ts, record_source):
    columns = ["customer_id", "customer_name", "credit_rating", "country"]
    df_raw = spark.createDataFrame(raw_data, columns)
    
    # Standardize business key (trim + uppercase) before hashing
    df_staged = (
        df_raw
        .withColumn("bk_clean", F.upper(F.trim(F.col("customer_id"))))
        # 1. Generate Hub Hash Key (SHA-256 of Business Key)
        .withColumn("hk_customer", F.sha2(F.col("bk_clean"), 256))
        # 2. Generate Satellite HashDiff (SHA-256 of concatenated descriptive attributes)
        .withColumn(
            "hashdiff",
            F.sha2(
                F.concat_ws(
                    "||",
                    F.coalesce(F.trim(F.col("customer_name")), F.lit("")),
                    F.coalesce(F.trim(F.col("credit_rating")), F.lit("")),
                    F.coalesce(F.trim(F.col("country")), F.lit(""))
                ),
                256
            )
        )
        .withColumn("load_dts", F.to_timestamp(F.lit(load_ts)))
        .withColumn("record_source", F.lit(record_source))
        .drop("bk_clean")
    )
    return df_staged

# Batch 1: Initial Day 1 Load
batch_1_data = [
    ("C-1001", "Anna Lindqvist", "AAA", "SE"),
    ("C-1002", "Erik Johansson", "AA", "SE")
]
# Convey current time
current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
df_stg_day1 = prepare_staging_data(batch_1_data, current_time, "CORE_BANKING")
display(df_stg_day1)
df_stg_day1.show()

In [ ]:
table_name="hub_customer"
table_path = f"{dv2_path}/{table_name}"
print(f"table_name: {table_name}")
print(f"dv2_path: {dv2_path}")
print(f"table_path: {table_path}")

In [ ]:
def merge_hub_external(df_source, table_name, hk_col):
    full_table_name = f"dv2_silver.{table_name}"
    table_path = f"{dv2_path}/{table_name}"
    df_dedup = df_source.dropDuplicates([hk_col])
    # Check physical path on disk rather than just the temporary catalog
    if not DeltaTable.isDeltaTable(spark, table_path):
        (
            df_dedup.write.format("delta").option("path", table_path).saveAsTable(full_table_name)
        )
    else:
        # Ensure table is registered in current session catalog, then MERGE
        spark.sql(f"CREATE TABLE IF NOT EXISTS {full_table_name} USING DELTA LOCATION '{table_path}'")
        dt = DeltaTable.forPath(spark, table_path)
        (
            dt.alias("t").merge(df_dedup.alias("s"), f"t.{hk_col} = s.{hk_col}")
            .whenNotMatchedInsertAll()
            .execute()
        )
    # Return the path so it can be used outside the function
    return table_path

# Load Customer Hub
table_path=merge_hub_external(df_stg_day1.select("hk_customer", "customer_id", "load_dts", "record_source"),"hub_customer","hk_customer")

"""
#The fun merge_hub missed to use table_path
def merge_hub(df_source, table_name, hk_col):
    full_table_name = f"dv2_silver.{table_name}"
    df_dedup = df_source.dropDuplicates([hk_col])
    
    if not spark.catalog.tableExists(full_table_name):
        df_dedup.write.format("delta").saveAsTable(full_table_name)
    else:
        dt = DeltaTable.forName(spark, full_table_name)
        (
            dt.alias("t")
            .merge(df_dedup.alias("s"), f"t.{hk_col} = s.{hk_col}")
            .whenNotMatchedInsertAll()
            .execute()
        )
"""

"""
#The fun load_hub_customer below missed to use extension saveAsTable(full_table_name)
def load_hub_customer(df_staged, target_path):
    df_hub_source = (
        df_staged
        .select("hk_customer", "customer_id", "load_dts", "record_source")
        .dropDuplicates(["hk_customer"])
    )
    
    if not DeltaTable.isDeltaTable(spark, target_path):
        df_hub_source.write.format("delta").save(target_path)
    else:
        hub_table = DeltaTable.forPath(spark, target_path)
        (
            hub_table.alias("target")
            .merge(
                df_hub_source.alias("source"),
                "target.hk_customer = source.hk_customer"
            )
            .whenNotMatchedInsertAll()
            .execute()
        )
"""
#load_hub_customer(df_stg_day1, hub_path)

print("--- HUB CUSTOMER (After Day 1) ---")
spark.read.format("delta").load(table_path).show(truncate=False)

In [ ]:
def merge_satellite_external(df_source, table_name, hk_col):
    full_table_name = f"dv2_silver.{table_name}"
    table_path = f"{dv2_path}/{table_name}"
    
    # 1. First-time load: Write Delta files to path and register external table
    if not DeltaTable.isDeltaTable(spark, table_path):
        (
            df_source.write.format("delta")
            .option("path", table_path)
            .saveAsTable(full_table_name)
        )
    else:
        # 2. Ensure table is registered in the current session catalog
        spark.sql(f"CREATE TABLE IF NOT EXISTS {full_table_name} USING DELTA LOCATION '{table_path}'")
        dt = DeltaTable.forPath(spark, table_path)
        
        # 3. Isolate only the latest historical row per hash key in the target Satellite
        window_spec = Window.partitionBy(hk_col).orderBy(F.col("load_dts").desc())
        df_latest_target = (
            dt.toDF()
            .withColumn("rn", F.row_number().over(window_spec))
            .filter(F.col("rn") == 1)
            .select(hk_col, "hashdiff")
        )
        
        # 4. Filter source to keep only new keys or rows with a changed hashdiff
        df_deltas = (
            df_source.alias("src")
            .join(
                df_latest_target.alias("tgt"),
                (F.col(f"src.{hk_col}") == F.col(f"tgt.{hk_col}")) & 
                (F.col("src.hashdiff") == F.col("tgt.hashdiff")),
                "left_anti"
            )
        )
        
        # 5. Insert true changes into the Satellite
        (
            dt.alias("t")
            .merge(
                df_deltas.alias("s"),
                f"t.{hk_col} = s.{hk_col} AND t.load_dts = s.load_dts"
            )
            .whenNotMatchedInsertAll()
            .execute()
        )
    # Return the path so it can be used outside the function
    return table_path

"""
#The fun load_sat_customer missed to use extension saveAsTable(full_table_name)
def load_sat_customer(df_staged, target_path):
    df_sat_source = df_staged.select(
        "hk_customer", "load_dts", "hashdiff", "record_source",
        "customer_name", "credit_rating", "country"
    )
    
    if not DeltaTable.isDeltaTable(spark, target_path):
        df_sat_source.write.format("delta").save(target_path)
    else:
        sat_table = DeltaTable.forPath(spark, target_path)
        
        # Identify only the LATEST record per hk_customer in the target Satellite
        window_spec = Window.partitionBy("hk_customer").orderBy(F.col("load_dts").desc())
        df_latest_target = (
            sat_table.toDF()
            .withColumn("rn", F.row_number().over(window_spec))
            .filter(F.col("rn") == 1)
            .select("hk_customer", "hashdiff")
        )
        
        # Keep only source rows that are either brand-new keys OR have a changed hashdiff
        df_deltas = (
            df_sat_source.alias("src")
            .join(
                df_latest_target.alias("tgt"),
                (F.col("src.hk_customer") == F.col("tgt.hk_customer")) & 
                (F.col("src.hashdiff") == F.col("tgt.hashdiff")),
                "left_anti"
            )
        )
        
        # Insert the true changes into the Satellite
        (
            sat_table.alias("target")
            .merge(
                df_deltas.alias("source"),
                "target.hk_customer = source.hk_customer AND target.load_dts = source.load_dts"
            )
            .whenNotMatchedInsertAll()
            .execute()
        )
"""

"""
#The fun merge_satellite below missed to use table_path
def merge_satellite(df_source, table_name, hk_col):
    full_table_name = f"dv2_silver.{table_name}"
    
    if not spark.catalog.tableExists(full_table_name):
        df_source.write.format("delta").saveAsTable(full_table_name)
    else:
        dt = DeltaTable.forName(spark, full_table_name)
        window_spec = Window.partitionBy(hk_col).orderBy(F.col("load_dts").desc())
        
        df_latest = (
            dt.toDF()
            .withColumn("rn", F.row_number().over(window_spec))
            .filter(F.col("rn") == 1)
            .select(hk_col, "hashdiff")
        )
        
        df_deltas = (
            df_source.alias("src")
            .join(
                df_latest.alias("tgt"),
                (F.col(f"src.{hk_col}") == F.col(f"tgt.{hk_col}")) & 
                (F.col("src.hashdiff") == F.col("tgt.hashdiff")),
                "left_anti"
            )
        )
        
        (
            dt.alias("t")
            .merge(
                df_deltas.alias("s"),
                f"t.{hk_col} = s.{hk_col} AND t.load_dts = s.load_dts"
            )
            .whenNotMatchedInsertAll()
            .execute()
        )
"""
#load_sat_customer(df_stg_day1, sat_path)
#load Customer Satellite
table_path=merge_satellite_external(df_stg_day1.select("hk_customer", "load_dts", "hashdiff", "record_source","customer_name", "credit_rating", "country"),
                         "sat_customer_details","hk_customer")
print("--- SATELLITE CUSTOMER (After Day 1) ---")
spark.read.format("delta").load(table_path).show(truncate=False)

In [ ]:
"""batch_1_data = [
    ("C-1001", "Anna Lindqvist", "AAA", "SE"),
    ("C-1002", "Erik Johansson", "AA", "SE")
]"""

batch_2_data = [
    ("C-1001", "Anna Lindqvist", "AAA", "SE"),   # Unchanged
    ("C-1002", "Erik Johansson", "BBB", "SE"),   # Credit rating changed!
    ("C-1003", "Sofia Berg", "A", "NO")          # Brand new customer
]

# Convey current time
current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
df_stg_day2 = prepare_staging_data(batch_2_data, current_time, "CORE_BANKING")
display(df_stg_day2)
df_stg_day2.show()

#load_hub_customer(df_stg_day2, hub_path) -- commented it as used merge_hub_external fun below to do the same
# Load Customer Hub
table_path=merge_hub_external(df_stg_day2.select("hk_customer", "customer_id", "load_dts", "record_source"),"hub_customer","hk_customer")
print("--- HUB CUSTOMER (After Day 2: 3 Unique Keys) ---")
spark.read.format("delta").load(table_path).orderBy("customer_id").show(truncate=False)

# 1. Check the exact table location and row count
spark.sql("DESCRIBE DETAIL dv2_silver.hub_customer").select("name", "location", "numFiles").show(truncate=False)
spark.table("dv2_silver.hub_customer").show(truncate=False)

# 2. Check the Delta audit log to see what the Day 2 MERGE actually did
dt = DeltaTable.forName(spark, "dv2_silver.hub_customer")
dt.history().select("version", "timestamp", "operation", "operationMetrics").show(truncate=False)

#load_sat_customer(df_stg_day2, sat_path)
#load Customer Satellite
table_path=merge_satellite_external(df_stg_day2.select("hk_customer", "load_dts", "hashdiff", "record_source","customer_name", "credit_rating", "country"),
                         "sat_customer_details","hk_customer")
print("--- SATELLITE CUSTOMER (After Day 2: 4 Historical Rows) ---")
spark.read.format("delta").load(table_path).orderBy("hk_customer", "load_dts").show(truncate=False)


In [ ]:
# 1. Create the schema
#spark.sql("DROP SCHEMA IF EXISTS dv2_bronze CASCADE")
#spark.sql("CREATE SCHEMA IF NOT EXISTS dv2_silver")
# 2. Register the folder path as external tables inside that schema
#spark.sql("DROP TABLE IF EXISTS dv2_silver.hub_customer")
#spark.sql(f""" CREATE TABLE IF NOT EXISTS dv2_silver.hub_customer USING DELTA  LOCATION '{hub_path}' """)
#spark.sql(f""" CREATE TABLE dv2_silver.sat_customer_details USING DELTA LOCATION '{sat_path}' """)

In [ ]:
def stage_loan_risk_feed(raw_rows, load_ts, record_source):
    cols = [
        "customer_id", "loan_account_number", "outstanding_principal",
        "days_past_due", "forbearance_flag", "ifrs9_stage", "impaired_flag"]
    df = spark.createDataFrame(raw_rows, cols)
    
    df_staged = (
        df
        .withColumn("bk_cust", F.upper(F.trim(F.col("customer_id"))))
        .withColumn("bk_loan", F.upper(F.trim(F.col("loan_account_number"))))
        # 1. Hub Hash Keys
        .withColumn("hk_customer", F.sha2(F.col("bk_cust"), 256))
        .withColumn("hk_loan_account", F.sha2(F.col("bk_loan"), 256))
        # 2. Link Hash Key (SHA-256 of combined business keys)
        .withColumn("hk_customer_loan", F.sha2(F.concat_ws("||", F.col("bk_cust"), F.col("bk_loan")), 256))
        # 3. Satellite HashDiff (SHA-256 of descriptive risk attributes)
        .withColumn(
            "hashdiff",
            F.sha2(
                F.concat_ws(
                    "||",
                    F.coalesce(F.col("outstanding_principal").cast("string"), F.lit("")),
                    F.coalesce(F.col("days_past_due").cast("string"), F.lit("")),
                    F.coalesce(F.trim(F.col("forbearance_flag")), F.lit("")),
                    F.coalesce(F.col("ifrs9_stage").cast("string"), F.lit("")),
                    F.coalesce(F.trim(F.col("impaired_flag")), F.lit(""))
                ),
                256
            )
        )
        .withColumn("load_dts", F.to_timestamp(F.lit(load_ts)))
        .withColumn("record_source", F.lit(record_source))
        .drop("bk_cust", "bk_loan")
    )
    return df_staged

# Month 1 Sample Data (Oct 2026)
month1_data = [
    ("C-1001", "LN-9001", 1500000.00, 0,  "N", 1, "N"),  # Performing mortgage
    ("C-1002", "LN-9002", 250000.00,  15, "N", 1, "N"),  # Slightly past due personal loan
    ("C-1003", "LN-9003", 800000.00,  45, "Y", 2, "N")   # Under forbearance, Stage 2
]

df_stg_m1 = stage_loan_risk_feed(month1_data, "2026-10-31 23:59:00", "CORE_LOANS")
df_stg_m1.show()

# Load Hub, Link, and Satellite for Month 1
# Load Loan Account Hub
table_path=merge_hub_external(df_stg_m1.select("hk_loan_account", "loan_account_number", "load_dts", "record_source"),
                              "hub_loan_account","hk_loan_account")

print("--- HUB Loan Account (After Month 1) ---")
spark.read.format("delta").load(table_path).show(truncate=False)

# Load Customer Loan Link
table_path=merge_hub_external(df_stg_m1.select("hk_customer_loan", "hk_customer", "hk_loan_account", "load_dts", "record_source"),
                              "link_customer_loan","hk_customer_loan")

print("--- Link Customer Loan (After Month 1) ---")
spark.read.format("delta").load(table_path).show(truncate=False)

# Load Loan Risk Status Satellite
table_path=merge_satellite_external(df_stg_m1.select("hk_loan_account", "load_dts", "hashdiff", "record_source",
                                 "outstanding_principal", "days_past_due", "forbearance_flag", "ifrs9_stage", "impaired_flag"),
                "sat_loan_risk_status","hk_loan_account")

print("--- Satellite Loan Risk Status (After Month 1) ---")
spark.read.format("delta").load(table_path).show(truncate=False)

print("Month 1 Data Vault load complete!")

In [ ]:
# Month 2 Sample Data (Nov 2026)
month2_data = [
    ("C-1001", "LN-9001", 1500000.00, 0,  "N", 1, "N"),  # Unchanged -> Ignored
    ("C-1002", "LN-9002", 250000.00,  95, "N", 3, "Y"),  # Deteriorated to Stage 3 Impaired -> New Satellite Row
    ("C-1003", "LN-9003", 800000.00,  45, "Y", 2, "N")   # Unchanged -> Ignored
]

df_stg_m2 = stage_loan_risk_feed(month2_data, "2026-11-30 23:59:00", "CORE_LOANS")
df_stg_m2.show()

# Load Hub, Link, and Satellite for Month 2
# Load Loan Account Hub
table_path=merge_hub_external(df_stg_m2.select("hk_loan_account", "loan_account_number", "load_dts", "record_source"),
                              "hub_loan_account","hk_loan_account")

print("--- HUB Loan Account (After Month 2) ---")
spark.read.format("delta").load(table_path).show(truncate=False)

# Load Customer Loan Link
table_path=merge_hub_external(df_stg_m2.select("hk_customer_loan", "hk_customer", "hk_loan_account", "load_dts", "record_source"),
                              "link_customer_loan","hk_customer_loan")

print("--- Link Customer Loan (After Month 2) ---")
spark.read.format("delta").load(table_path).show(truncate=False)

# Load Loan Risk Status Satellite
table_path=merge_satellite_external(df_stg_m2.select("hk_loan_account", "load_dts", "hashdiff", "record_source",
                                 "outstanding_principal", "days_past_due", "forbearance_flag", "ifrs9_stage", "impaired_flag"),
                "sat_loan_risk_status","hk_loan_account")

print("--- Satellite Loan Risk Status (After Month 2) ---")
spark.read.format("delta").load(table_path).show(truncate=False)

# Query the full audit trail across Hub, Link, and Satellite
spark.sql("""
    SELECT 
        h.loan_account_number,
        s.load_dts,
        s.outstanding_principal,
        s.days_past_due,
        s.forbearance_flag,
        s.ifrs9_stage,
        s.impaired_flag
    FROM dv2_silver.hub_loan_account h
    JOIN dv2_silver.sat_loan_risk_status s 
        ON h.hk_loan_account = s.hk_loan_account
    ORDER BY h.loan_account_number, s.load_dts
""").show(truncate=False)

In [ ]:
def stage_collateral_feed(raw_rows, load_ts, record_source):
    cols = ["loan_account_number", "collateral_id", "collateral_type", "market_valuation", "haircut_pct"]
    df = spark.createDataFrame(raw_rows, cols)
    
    return (
        df
        .withColumn("bk_loan", F.upper(F.trim(F.col("loan_account_number"))))
        .withColumn("bk_coll", F.upper(F.trim(F.col("collateral_id"))))
        .withColumn("hk_loan_account", F.sha2(F.col("bk_loan"), 256))
        .withColumn("hk_collateral", F.sha2(F.col("bk_coll"), 256))
        .withColumn("hk_loan_collateral", F.sha2(F.concat_ws("||", F.col("bk_loan"), F.col("bk_coll")), 256))
        # Calculate cleansed forced-sale value after regulatory haircut
        .withColumn("cleansed_collateral_value", F.round(F.col("market_valuation") * (1 - F.col("haircut_pct") / 100.0), 2))
        .withColumn(
            "hashdiff",
            F.sha2(
                F.concat_ws(
                    "||",
                    F.col("collateral_type"),
                    F.col("market_valuation").cast("string"),
                    F.col("haircut_pct").cast("string")
                ),
                256
            )
        )
        .withColumn("load_dts", F.to_timestamp(F.lit(load_ts)))
        .withColumn("record_source", F.lit(record_source))
        .drop("bk_loan", "bk_coll")
    )

# Sample Collateral Data (Real Estate & Cash Pledge)
collateral_data = [
    ("LN-9001", "COL-501", "RESIDENTIAL_PROPERTY", 2200000.00, 20.0),  # 20% haircut -> 1,760,000 cleansed value
    ("LN-9003", "COL-502", "COMMERCIAL_REAL_ESTATE", 1000000.00, 35.0) # 35% haircut -> 650,000 cleansed value
]

df_coll = stage_collateral_feed(collateral_data, "2026-11-30 23:59:00", "COLLATERAL_SYS")
df_coll.show()

# Load Collateral Hub
table_path=merge_hub_external(df_coll.select("hk_collateral", "collateral_id", "load_dts", "record_source"),
                              "hub_collateral","hk_collateral")

print("--- HUB Collateral (After Month 1) ---")
spark.read.format("delta").load(table_path).show(truncate=False)

# Load Loan Collateral Link
table_path=merge_hub_external(df_coll.select("hk_loan_collateral", "hk_loan_account", "hk_collateral", "load_dts", "record_source"),
                              "link_loan_collateral","hk_loan_collateral")

print("--- Link Loan Collateral (After Month 1) ---")
spark.read.format("delta").load(table_path).show(truncate=False)

# Load Collateral Valuation Satellite
table_path=merge_satellite_external(df_coll.select("hk_collateral", "load_dts", "hashdiff", "record_source", "collateral_type", "market_valuation", "haircut_pct", "cleansed_collateral_value"),
                                    "sat_collateral_valuation","hk_collateral")

print("--- Satellite Loan Risk Status (After Month 1) ---")
spark.read.format("delta").load(table_path).show(truncate=False)

print("Collateral Hub, Link, and Satellite created successfully!")
spark.sql("SELECT * FROM dv2_silver.sat_collateral_valuation").show(truncate=False)

In [ ]:
# 1. List all available catalogs (similar to Unity Catalog top-level view)
spark.sql("SHOW CATALOGS").show()
# 2. List all schemas / databases
spark.sql("SHOW SCHEMAS").show()
# 3. List all tables inside your schema
spark.sql("SHOW TABLES IN dv2_silver").show()
# 4. View column names, data types, and comments (like Databricks Schema tab)
spark.sql("DESCRIBE TABLE EXTENDED dv2_silver.hub_customer").show(truncate=False)
spark.sql("DESCRIBE TABLE EXTENDED dv2_silver.hub_loan_account").show(truncate=False)
spark.sql("DESCRIBE TABLE EXTENDED dv2_silver.hub_collateral").show(truncate=False)
spark.sql("DESCRIBE TABLE EXTENDED dv2_silver.link_customer_loan").show(truncate=False)
spark.sql("DESCRIBE TABLE EXTENDED dv2_silver.link_loan_collateral").show(truncate=False)
spark.sql("DESCRIBE TABLE EXTENDED dv2_silver.sat_customer_details").show(truncate=False)
spark.sql("DESCRIBE TABLE EXTENDED dv2_silver.sat_loan_risk_status").show(truncate=False)
spark.sql("DESCRIBE TABLE EXTENDED dv2_silver.sat_collateral_valuation").show(truncate=False)

In [ ]:
def show_catalog_explorer():
    print("🗄️  SPARK CATALOG EXPLORER")
    print("=" * 55)
    
    # Iterate through catalogs
    catalogs = [row.catalog for row in spark.sql("SHOW CATALOGS").collect()]
    for cat in catalogs:
        print(f"📁 Catalog: {cat}")
        spark.catalog.setCurrentCatalog(cat)
        
        # Iterate through schemas (databases)
        databases = spark.catalog.listDatabases()
        for db in databases:
            print(f"  └── 📂 Schema: {db.name}")
            
            # Iterate through tables in each schema
            tables = spark.catalog.listTables(db.name)
            if not tables:
                print(f"        └── (No tables registered)")
            for tbl in tables:
                print(f"        └── 📄 Table: {tbl.name} [{tbl.tableType}]")
                
                # List columns and data types
                #columns = spark.catalog.listColumns(tbl.name, db.name)
                columns = spark.catalog.listColumns(f"{db.name}.{tbl.name}")
                for col in columns:
                    pk_flag = "🔑" if col.isPartition else "  "
                    print(f"              {pk_flag} {col.name}: {col.dataType} (nullable={col.nullable})")
    print("=" * 55)

# Run the explorer
show_catalog_explorer()

In [ ]:
spark.sql(f"""
describe history dv2_silver.hub_customer ;""")